# Circuit Discovery: Finding Critical Reasoning Components

This notebook demonstrates how to discover which model components (attention heads, layers) are **critical** for reasoning tasks.

**Key concepts:**
- **Activation Patching**: Replace clean activations with corrupted ones to measure causal effects
- **Circuit**: Minimal set of components that preserve model performance
- **Information Flow**: Track how information propagates from premises to conclusions
- **Attribution**: Identify which inputs/layers contribute most to outputs

**What we'll learn:**
1. Run activation patching experiments
2. Discover reasoning circuits
3. Visualize information flow graphs
4. Compute token and layer attributions

## Setup

In [1]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from interpretability import load_model
from interpretability.extraction import extract_attention_patterns, extract_hidden_states
from interpretability.analysis import (
    activation_patching_experiment,
    ablate_components,
    discover_reasoning_circuit,
    compute_information_flow,
    trace_reasoning_path,
    detect_bottleneck_layers,
    identify_critical_tokens,
    compute_token_attribution,
    compute_layer_contribution,
)
from interpretability.visualization import (
    build_computation_graph,
    render_graph_interactive,
    plot_circuit_overview,
)

print("✅ Imports successful!")

Matplotlib is building the font cache; this may take a moment.
/Users/wkang/code1/reasoning_explore/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-19 17:04:03.947 | DEBUG    | interpretability.models.registry:register:54 - Registered loader 'deepseek-r1-distill': DeepSeekLoader
2026-04-19 17:04:03.947 | DEBUG    | interpretability.models.registry:register:54 - Registered loader 'deepseek': DeepSeekLoader
2026-04-19 17:04:03.948 | DEBUG    | interpretability.models.registry:add_alias:65 - Added alias 'deepseek-1.5b' -> 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
2026-04-19 17:04:03.948 | DEBUG    | interpretability.models.registry:add_alias:65 - Added alias 'deepseek-7b' -> 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
2026-04-19 17:04:03.948 | DEBUG    | interpretability.models.registry:add_a

✅ Imports successful!


## 1. Load Model and Prepare Examples

In [2]:
# Load model (adjust device/quantization for your platform)
print("Loading model...")
model = load_model(
    "deepseek-1.5b",
    device="cpu",  # or "cuda", "mps"
    quantization=None  # or "4bit" on Linux/Windows with GPU
)

print(f"✅ Model loaded: {model.num_layers} layers, {model.num_attention_heads} heads")
print(f"   Memory: {model.memory_stats()['param_size_mb']:.1f} MB")

2026-04-19 17:04:07.004 | DEBUG    | interpretability.models.registry:get_loader:93 - Matched 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B' to loader 'deepseek-r1-distill'
2026-04-19 17:04:07.004 | INFO     | interpretability.models.deepseek:load:61 - Loading DeepSeek model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
2026-04-19 17:04:07.015 | DEBUG    | interpretability.models.deepseek:get_recommended_quantization:216 - macOS detected - recommending no quantization
2026-04-19 17:04:07.015 | INFO     | interpretability.core.model_wrapper:_load_model:142 - Loading model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B...
2026-04-19 17:04:07.015 | DEBUG    | interpretability.core.model_wrapper:_load_model:156 - Using 'eager' attention implementation for interpretability


Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 249.89it/s]
2026-04-19 17:04:08.939 | DEBUG    | interpretability.core.model_wrapper:_load_model:176 - Enabled output_attentions in model config
2026-04-19 17:04:08.939 | DEBUG    | interpretability.core.model_wrapper:_load_model:181 - Enabled output_hidden_states in model config
2026-04-19 17:04:08.940 | INFO     | interpretability.core.model_wrapper:_load_model:183 - Loaded model on cpu with no quantization
2026-04-19 17:04:09.983 | DEBUG    | interpretability.core.model_wrapper:_load_tokenizer:205 - Loaded tokenizer for deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
2026-04-19 17:04:09.984 | DEBUG    | interpretability.core.model_wrapper:_initialize_metadata:233 - Model metadata: 28 layers, 12 heads, 1536 hidden size
2026-04-19 17:04:09.984 | INFO     | interpretability.core.activation_cache:__init__:55 - Initialized ActivationCache with max size 1000 MB
2026-04-19 17:04:09.985 | INF

AttributeError: 'ModelWrapper' object has no attribute 'num_layers'

In [ ]:
# Define clean (correct) and corrupted (incorrect) reasoning examples
clean_prompt = "What is 8 + 7? Let me think. 8 + 7 = 15. The answer is 15."
corrupted_prompt = "What is 8 + 7? Let me think. 8 + 7 = 16. The answer is 16."

print(f"Clean prompt:     {clean_prompt}")
print(f"Corrupted prompt: {corrupted_prompt}")

# Tokenize
clean_ids = model.tokenize(clean_prompt, move_to_device=True)['input_ids']
corrupted_ids = model.tokenize(corrupted_prompt, move_to_device=True)['input_ids']
tokens = model.tokenizer.convert_ids_to_tokens(clean_ids[0])

print(f"\nTokens ({len(tokens)}): {tokens}")

# Get target token (correct answer "15")
target_token_id = model.tokenizer.encode("15", add_special_tokens=False)[0]
print(f"Target token ID: {target_token_id} ('{model.tokenizer.decode([target_token_id])}')") 

## 2. Activation Patching Experiment

**Goal**: Identify which components are critical for the correct answer.

**Method**: For each component (attention head, MLP):
1. Run clean input through model
2. Patch in corrupted activation for that component
3. Measure change in target token logit
4. Large change = component is important

In [ ]:
# Run activation patching on a subset of layers (for speed)
test_layers = [10, 15, 20, 25]  # Test middle and late layers

print(f"Running activation patching on layers {test_layers}...")
print("This may take a few minutes...\n")

patching_results = activation_patching_experiment(
    model,
    clean_ids,
    corrupted_ids,
    target_token=target_token_id,
    layers=test_layers,
    component_types=["attention"],  # Test attention heads only
    position=-1,  # Last token position
    show_progress=True,
)

print(f"\n✅ Patching complete!")
print(f"   Clean logit: {patching_results.clean_logit:.3f}")
print(f"   Corrupted logit: {patching_results.corrupted_logit:.3f}")
print(f"   Tested {len(patching_results.components)} components")

In [ ]:
# Show top important components
top_components = patching_results.get_top_components(k=10)
print("\n🔍 Top 10 Most Important Components:")
print(top_components.to_string(index=False))

In [ ]:
# Visualize patching effects
import plotly.express as px

df = patching_results.to_dataframe()

fig = px.scatter(
    df,
    x="layer",
    y="head",
    size=df["patching_effect"].abs(),
    color="patching_effect",
    color_continuous_scale="RdBu",
    title="Activation Patching Effects",
    labels={"patching_effect": "Patching Effect"},
    hover_data=["component_name", "logit_diff"],
)

fig.update_layout(width=800, height=600)
fig.show()

## 3. Discover Reasoning Circuit

Now let's automatically discover the **minimal circuit** that preserves correct reasoning.

In [ ]:
# Discover circuit (keep components with >10% effect)
print("Discovering reasoning circuit...\n")

circuit = discover_reasoning_circuit(
    model,
    clean_prompt=clean_prompt,
    corrupted_prompt=corrupted_prompt,
    target_token=target_token_id,
    threshold=0.1,  # Keep components with >10% patching effect
    method="patching",
    layers=test_layers,
    show_progress=True,
)

print(f"\n✅ Circuit discovered!")
print(f"   Components: {len(circuit.components)}")
print(f"   Discovery method: {circuit.discovery_method}")
print(f"   Threshold: {circuit.threshold}")

In [ ]:
# Show circuit components
print("\n🔧 Circuit Components:")
for i, (comp, importance) in enumerate(zip(circuit.components, circuit.importance_scores)):
    print(f"  {i+1}. {comp.name}: {importance:.3f}")

In [ ]:
# Visualize circuit
fig = plot_circuit_overview(
    circuit,
    num_layers=model.num_layers,
    num_heads=model.num_attention_heads,
)

fig.show()

## 4. Information Flow Analysis

Track how information flows from premise tokens to the answer.

In [ ]:
# Extract attention for flow analysis
print("Extracting attention patterns...")
attention = extract_attention_patterns(model, clean_ids)
print(f"✅ Extracted attention: {attention.patterns.shape}")

# Build flow graph
print("\nBuilding information flow graph...")
flow_graph = compute_information_flow(
    attention,
    aggregation="max",  # Max attention across heads
    threshold=0.05,  # Minimum attention to include
)

print(f"✅ Flow graph built:")
print(f"   Nodes: {len(flow_graph.graph.nodes)}")
print(f"   Edges: {len(flow_graph.graph.edges)}")

In [ ]:
# Find reasoning paths from input numbers to answer
# Identify token positions for "8" and "7"
source_positions = []
for i, token in enumerate(tokens):
    if '8' in token or '7' in token:
        source_positions.append(i)
        print(f"Source token at position {i}: '{token}'")

# Target: last token (answer)
target_position = len(tokens) - 1
print(f"Target token at position {target_position}: '{tokens[target_position]}'")

# Trace paths
print("\nTracing reasoning paths...")
paths = trace_reasoning_path(
    flow_graph,
    source_tokens=source_positions[:2],  # First two number tokens
    target_token=target_position,
    max_paths=5,
)

print(f"\n🛤️  Found {len(paths)} reasoning paths:")
for i, path in enumerate(paths[:3]):
    print(f"\n  Path {i+1} (flow={path.total_flow:.4f}):")
    print(f"    {path}")

In [ ]:
# Detect bottleneck layers (where information compresses)
print("Detecting bottleneck layers...\n")
bottlenecks = detect_bottleneck_layers(
    flow_graph,
    method="entropy",  # Low entropy = focused attention
)

print("🔴 Top 5 Bottleneck Layers:")
for layer, score in bottlenecks[:5]:
    print(f"  Layer {layer:2d}: {score:.3f}")

In [ ]:
# Identify critical tokens for the answer
critical_tokens = identify_critical_tokens(
    flow_graph,
    target_token=target_position,
    top_k=10,
)

print("\n⭐ Top 10 Critical Tokens:")
for layer, pos, flow in critical_tokens:
    print(f"  Layer {layer:2d}, Position {pos:2d} ('{tokens[pos]}'): {flow:.4f}")

## 5. Visualize Computation Graph

Create an interactive visualization of information flow through the circuit.

In [ ]:
# Build computation graph (subset of layers for clarity)
comp_graph = build_computation_graph(
    flow_graph,
    circuit=circuit,
    layers_to_show=[15, 20, 25],  # Show a few layers
    min_weight=0.1,  # Only strong edges
)

print(f"Computation graph:")
print(f"  Nodes: {len(comp_graph.nodes)}")
print(f"  Edges: {len(comp_graph.edges)}")

# Render interactive graph
fig = render_graph_interactive(
    comp_graph,
    layout="hierarchical",
    title="Reasoning Circuit - Information Flow",
)

fig.show()

## 6. Token Attribution

Which input tokens are most important for the answer?

In [ ]:
# Compute token attribution
print("Computing token attribution...\n")
attribution = compute_token_attribution(
    model,
    clean_ids,
    target_token=target_token_id,
    method="gradient",  # or "attention" for gradient-free
)

print("✅ Attribution computed!")
print("\n📊 Top 10 Most Important Input Tokens:")
print(attribution.get_top_tokens(10).to_string(index=False))

In [ ]:
# Visualize attribution
from interpretability.analysis import visualize_token_attribution

print("\nToken Attribution Visualization:")
print("=" * 60)
print(visualize_token_attribution(attribution, output_format="text"))

In [ ]:
# Plot attribution as bar chart
import plotly.graph_objects as go

df = attribution.to_dataframe()

fig = go.Figure(data=[
    go.Bar(
        x=df['token'],
        y=df['attribution'],
        text=df['attribution'].round(3),
        textposition='auto',
    )
])

fig.update_layout(
    title="Token Attribution for Answer Token",
    xaxis_title="Token",
    yaxis_title="Attribution Score",
    width=1000,
    height=500,
)

fig.show()

## 7. Layer Contribution Analysis

Which layers contribute most to the final prediction?

In [ ]:
# Compute layer contributions
print("Computing layer contributions...\n")
layer_contrib = compute_layer_contribution(
    model,
    clean_ids,
    target_token=target_token_id,
    method="norm",  # Hidden state norm as proxy
)

print("✅ Layer contributions computed!")

In [ ]:
# Plot layer contributions
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(layer_contrib.layer_contributions))),
    y=layer_contrib.layer_contributions,
    mode='lines+markers',
    name='Layer Contribution',
    line=dict(width=2),
    marker=dict(size=8),
))

fig.update_layout(
    title="Layer-wise Contribution to Answer",
    xaxis_title="Layer",
    yaxis_title="Contribution (Hidden State Norm)",
    width=900,
    height=500,
)

fig.show()

## Summary

In this notebook, we learned how to:

1. ✅ **Activation Patching**: Identify critical components by patching corrupted activations
2. ✅ **Circuit Discovery**: Find minimal set of components for correct reasoning
3. ✅ **Information Flow**: Track how information propagates through layers
4. ✅ **Reasoning Paths**: Trace multi-hop paths from premises to conclusions
5. ✅ **Bottlenecks**: Detect layers where information compresses
6. ✅ **Token Attribution**: Identify which inputs matter most
7. ✅ **Layer Contribution**: Measure how much each layer contributes

## Next Steps

- Try different reasoning tasks (logic, math, multi-hop)
- Compare circuits across easy vs hard problems
- Test circuit transfer: does a circuit for addition work for subtraction?
- Explore other attribution methods (integrated gradients, attention rollout)
- Build your own circuit discovery algorithms!